In [ ]:
using MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8433*2*pi
ω_b_0 = 7.1515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.1123
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 36.924*2*pi#*(rand(1)[1]/4+0.875)


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5013#0.6013
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.124276*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
(ω_b_0 - ω_a)/2pi

In [ ]:
ω_p /2pi

In [ ]:
(ω_a + ω_p)/2pi

# First lobe

In [ ]:
function output_notebook(ϵ_0_input::Float64,(re_a_0_input, im_a_0_input)::Tuple{Float64,Float64})
    t_max = 60/κ_b
    t_min = 0.0
    t_span = (t_min, t_max)

    re_a_0, im_a_0 = re_a_0_input, im_a_0_input
    ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
    ϕ_b_0, q_b_0=0.0, 0.0

    system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
    environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

    system_prepared = prepare(system)
    environment_prepared = prepare(system,environment)

    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0_input, ω_p, pump_phase)
    drive_prepared = prepare(drive)

    dt = get_dt(system, environment)
    u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

    p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

    t_save = 30/κ_b:dt:60/κ_b
    ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!, t_save = t_save)
    return(ϕ_a, ϕ_b, q_a, q_b, t_list)
end

In [ ]:
ϕ_a, ϕ_b, q_a, q_b, t_list = output_notebook(0.5013,(10.0, -4.0))

In [ ]:
ω =  ω_a
Re = ϕ_a / sqrt(2*Z_a)
Im = q_a * sqrt(Z_a/2);
Re_a =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_a =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
plot(freqs, abs.(F), linewidth=.1, label="")
plot!(freqs, 300 ./freqs, ls=:dash, color=Perso_palette_hex[5], label=L"1/f"*" noise")
plot!(xlims=(0.5,580), ylims=(0.01,1e6), 
yaxis=:log, xaxis=:log,
xlabel="Frequency (GHz)", ylabel="|FFT|")

In [ ]:
abs(freqs[argmax(abs.(F))])

In [ ]:
power_density = abs.(F).^2
power_density_normalized = power_density ./sum(power_density)
Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))

# Coherent state convergence

In [ ]:
ϕ_a, ϕ_b, q_a, q_b, t_list = output_notebook(0.5,(0.0, 0.0))

ω =  ω_a
Re = ϕ_a / sqrt(2*Z_a)
Im = q_a * sqrt(Z_a/2);
Re_a =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_a =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);

In [ ]:
Re_a[end], Im_a[end]

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
plot(freqs, abs.(F), linewidth=.1, label="")
plot!(freqs, 12 ./freqs, ls=:dash, color=Perso_palette_hex[5], label=L"1/f"*" noise")
plot!(xlims=(0.5,580), ylims=(0.01,1e6), 
yaxis=:log, xaxis=:log,
xlabel="Frequency (GHz)", ylabel="|FFT|")

In [ ]:
abs(freqs[argmax(abs.(F))])

In [ ]:
power_density = abs.(F[div(length(freqs),2)+2:end]).^2
power_density_normalized = power_density ./sum(power_density)
Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))

In [ ]:
sin_arg =@. (ϕ_a + ϕ_b)/ϕ_quantum
plot(t_list, sin_arg)

# Second lobe

In [ ]:
ϕ_a, ϕ_b, q_a, q_b, t_list = output_notebook(0.58,(-5.0, 1.0))

ω =  ω_a
Re = ϕ_a / sqrt(2*Z_a)
Im = q_a * sqrt(Z_a/2);
Re_a =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_a =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);

In [ ]:
Re_a[end], Im_a[end]

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
plot(freqs, abs.(F), linewidth=.1, label="")
plot!(freqs, 330 ./freqs, ls=:dash, color=Perso_palette_hex[5], label=L"1/f"*" noise")
plot!(xlims=(0.5,580), ylims=(0.01,1e6), 
yaxis=:log, xaxis=:log,
xlabel="Frequency (GHz)", ylabel="|FFT|")

In [ ]:
abs(freqs[argmax(abs.(F))])

In [ ]:
F[argmax(abs.(F))]=1

In [ ]:
abs(freqs[argmax(abs.(F))])

In [ ]:
power_density = abs.(F[div(length(freqs),2)+2:end]).^2
power_density_normalized = power_density ./sum(power_density)
Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))

In [ ]:
sin_arg =@. (ϕ_a + ϕ_b)/ϕ_quantum
plot(t_list, sin_arg)

# Chaos

In [ ]:
ϕ_a, ϕ_b, q_a, q_b, t_list = output_notebook(1.0,(0.0, 0.0))

ω =  ω_a
Re = ϕ_a / sqrt(2*Z_a)
Im = q_a * sqrt(Z_a/2);
Re_a =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_a =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
plot(freqs, abs.(F), linewidth=.1, label="")
plot!(freqs, 570 ./freqs, ls=:dash, color=Perso_palette_hex[5], label=L"1/f"*" noise")
plot!(xlims=(0.1,580), ylims=(1,1e6),
    yaxis=:log, xaxis=:log,
    xlabel = "Frequency (GHz)", ylabel="|FFT|")

In [ ]:
abs(freqs[argmax(abs.(F))])

In [ ]:
power_density = abs.(F[div(length(freqs),2)+2:end]).^2
power_density_normalized = power_density ./sum(power_density)
Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))

In [ ]:
sin_arg =@. (ϕ_a + ϕ_b)/ϕ_quantum
plot(t_list, sin_arg)

# Main Code

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8433*2*pi+0.1
ω_b_0 = 7.1515*2*pi+0.1

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.1123
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 36.924*2*pi#*(rand(1)[1]/4+0.875)


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5013#0.6013
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.124276*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
ϵ_0_list = vcat(0.5:.05:1,1:.15:0.6pi);
im_a_0_list = -6:0.5:6;

In [ ]:
t_max = 60/κ_b
t_min = 0.0
t_span = (t_min, t_max)
f_peak_matrix = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))
for (i,ϵ_0) in ProgressBar(enumerate(ϵ_0_list))
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.124276*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    for (j,im_a_0) in enumerate(im_a_0_list)

        re_a_0, im_a_0 = 0, im_a_0
        ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
        ϕ_b_0, q_b_0=0.0, 0.0

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
        environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment)

        dt = get_dt(system, environment)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        t_save = 30/κ_b:dt:60/κ_b
        ϕ_a, _, q_a, _, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!, t_save = t_save)

        ω =  ω_a
        Re = ϕ_a / sqrt(2*Z_a)
        Im = q_a * sqrt(Z_a/2);
        Re_a =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
        Im_a =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list)
        F, freqs = FFT_perso(Re_a.-sum(Re_a)/length(Re_a), t_list)
        f_peak = abs(freqs[argmax(abs.(F))])
        f_peak_matrix[j,i] = f_peak
    end
end
        

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,f_peak_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\n"*L"f_{peak}"*" (GHz)", title = "Mean field ",
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

# Shannon entropy

In [ ]:
t_max = 60/κ_b
t_min = 0.0
t_span = (t_min, t_max)
Shannon_entropy_matrix = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))
for (i,ϵ_0) in ProgressBar(enumerate(ϵ_0_list))
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.124276*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    for (j,im_a_0) in enumerate(im_a_0_list)

        re_a_0, im_a_0 = 0, im_a_0
        ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
        ϕ_b_0, q_b_0=0.0, 0.0

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
        environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment)

        dt = get_dt(system, environment)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        t_save = 30/κ_b:dt:60/κ_b
        ϕ_a, _, _, _, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!, t_save = t_save)

        F, freqs = FFT_perso(ϕ_a, t_list)
        power_density = abs.(F).^2
        power_density_normalized = power_density ./sum(power_density)
        Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
        Shannon_entropy_matrix[j,i] = Shannon_entropy
    end
end
        

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Shannon_entropy_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\n"*"Shannon entropy", title = "Mean field ",
    clims = (0, maximum(Shannon_entropy_matrix)),
    right_margin=10mm)
#savefig("../../../figures/chaos/Shannon_entropy_02_12_2025.pdf")

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Shannon_entropy_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\n"*"Shannon entropy", title = "Mean field ",
    clims = (0, maximum(Shannon_entropy_matrix)),
    right_margin=10mm)
#savefig("../../../figures/chaos/Shannon_entropy_02_12_2025.pdf")

# Lyapunov exponent

In [ ]:
t_max = 10/κ_b
t_min = 0.0
t_span = (t_min, t_max)
t_w = 30/κ_b
Lyap_coef_chaos = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))
for (i,ϵ_0) in ProgressBar(enumerate(ϵ_0_list))
    g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
    ϵ_d=10.124276*g2
    drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    for (j,im_a_0) in enumerate(im_a_0_list)

        re_a_0, im_a_0 = 0, im_a_0
        ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
        ϕ_b_0, q_b_0=0.0, 0.0

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
        environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment)

        dt = get_dt(system, environment)
        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
        u_0_new = vcat(u_0,u_dot_0);

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    end
end
        

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Lyap_coef_chaos)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\nLyapunov exp.", title = "Mean field ",
    right_margin=10mm)
#savefig("../../figures/Two-photon/heatmap_lyapunov_coef.pdf")

In [ ]:
Lyap_coef_chaos_sign = map(x -> x>0 ? 1 : -1, Lyap_coef_chaos);

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Lyap_coef_chaos_sign, cmap=:bluesreds)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\nLyapunov exp.", title = "Mean field ",
    right_margin=10mm)
#savefig("../../../figures/chaos/Lyapunov_comparison_spectrum_02_12_2025.pdf")

# Averaging

In [ ]:
n_iter = 10
offset = range(start=-1, stop=1, length=n_iter)

ω_a_list = 3.8433*2*pi.+offset
ω_b_list = 7.1515*2*pi.+offset

t_max = 60/κ_b
t_min = 0.0
t_span = (t_min, t_max)
Shannon_entropy_matrix = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))

Threads.@threads for index in 1:n_iter
    ω_a = ω_a_list[index]
    ω_b = ω_b_list[index]

    φ_a = 0.1123
    Z_a = 2*ϕ_quantum^2*φ_a^2
    C_a_inv = Z_a*ω_a
    E_Ca = C_a_inv/2

    φ_b = 0.204
    Z_b = 2*ϕ_quantum^2*φ_b^2
    C_b_inv = Z_b*ω_b
    E_Cb = C_b_inv/2


    ##Drive
    ω_d=ω_b
    ω_p=2ω_a-ω_d
    for (i,ϵ_0) in enumerate(ϵ_0_list)
        g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
        ϵ_d=10.124276*g2
        drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
        drive_prepared = prepare(drive)
        for (j,im_a_0) in enumerate(im_a_0_list)

            re_a_0, im_a_0 = 0, im_a_0
            ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
            ϕ_b_0, q_b_0=0.0, 0.0

            system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
            environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment)

            dt = get_dt(system, environment)
            u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            t_save = 30/κ_b:dt:60/κ_b
            ϕ_a, _, _, _, t_list = default_solver((u_dot_0, u_0), t_span, p, dt; equation=default_equation!, t_save = t_save)

            F, freqs = FFT_perso(ϕ_a, t_list)
            power_density = abs.(F).^2
            power_density_normalized = power_density ./sum(power_density)
            Shannon_entropy = -sum(power_density_normalized.*log.(power_density_normalized))
            Shannon_entropy_matrix[j,i] = Shannon_entropy./n_iter
        end
    end
end       

In [ ]:
g2_list = E_J.*besselj.(1,ϵ_0_list).*φ_a^2 .*φ_b

heatmap(g2_list/2/pi*1e3,im_a_0_list,Shannon_entropy_matrix)
plot!(xlabel=L"g_2/2\pi"*" (MHz)", ylabel=L"\Im[a(0)]",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(g2_list/2pi*1e3),maximum(g2_list/2pi*1e3)),ylims=(minimum(im_a_0_list),maximum(im_a_0_list)),
    colorbar_title="\n"*"Shannon entropy", title = "Mean field average over n=$n_iter frequencies",
    clims = (0, maximum(Shannon_entropy_matrix)),
    right_margin=10mm)
#savefig("../../../figures/chaos/Shannon_entropy_average_15_12_2025.pdf")

In [ ]:
t_max = 10/κ_b
t_min = 0.0
t_span = (t_min, t_max)
t_w = 30/κ_b
Lyap_coef_chaos = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))

n_iter = 10
offset = range(start=-1, stop=1, length=n_iter)

ω_a_list = 3.8433*2*pi.+offset
ω_b_list = 7.1515*2*pi.+offset

t_max = 60/κ_b
t_min = 0.0
t_span = (t_min, t_max)
Shannon_entropy_matrix = Array{Float64}(undef, length(im_a_0_list), length(ϵ_0_list))

for index in 1:n_iter
    ω_a = ω_a_list[index]
    ω_b = ω_b_list[index]

    φ_a = 0.1123
    Z_a = 2*ϕ_quantum^2*φ_a^2
    C_a_inv = Z_a*ω_a
    E_Ca = C_a_inv/2

    φ_b = 0.204
    Z_b = 2*ϕ_quantum^2*φ_b^2
    C_b_inv = Z_b*ω_b
    E_Cb = C_b_inv/2


    ##Drive
    ω_d=ω_b
    ω_p=2ω_a-ω_d
    for (i,ϵ_0) in ProgressBar(enumerate(ϵ_0_list))
        g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
        ϵ_d=10.124276*g2
        drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
        drive_prepared = prepare(drive)
        for (j,im_a_0) in enumerate(im_a_0_list)

            re_a_0, im_a_0 = 0, im_a_0
            ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
            ϕ_b_0, q_b_0=0.0, 0.0

            system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
            environment = MeanField(κ_a, κ_b) #OneChain(n, chain_params...)

            system_prepared = prepare(system)
            environment_prepared = prepare(system,environment)

            dt = get_dt(system, environment)
            u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
            u_0_new = vcat(u_0,u_dot_0);

            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            problem = CoupledODEs(equation_1ODE!, u_0_new, p)

            Lyap_coef_chaos[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        end
    end
end       